# LlamaIndex: a hands-on implementation

**Track:** Frameworks · **Guide:** [`057-llamaindex.md`](../guides/concepts/057-llamaindex.md)

LlamaIndex provides data connectors and indexing/query abstractions to organize private data for LLM applications.

> **Intuition:** It is an indexing desk that turns a pile of documents into a structured, searchable reference collection.

This lesson builds a small working example in stages: prepare inputs, implement
the core idea, run it on a realistic toy case, and inspect its output and limits.
It runs offline unless a cell is explicitly marked as an optional provider call.

```mermaid
flowchart LR
    A[Prepare inputs] --> B[Implement LlamaIndex]
    B --> C[Run on example]
    C --> D[Evaluate and inspect]
```

**Case study:** Ingest versioned internal policy documents, keep source metadata, and evaluate which chunks answer representative queries.

Run the cells from top to bottom. To install the notebook environment, follow
[`guides/SETUP.md`](../guides/SETUP.md). The toy data and implementation are for
learning, not a production quality, safety, or security guarantee.


## 1. Split source documents into chunks

LlamaIndex structures data for retrieval. This local example demonstrates chunk IDs and source metadata without an external model.


In [ ]:
documents = [
    {"id": "policy-7", "text": "Unused products may be returned within 30 days. Keep the receipt."},
    {"id": "support-2", "text": "Support is open Monday through Friday."},
]

def chunk_text(text, size=7, overlap=2):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        chunks.append(" ".join(words[start:start + size]))
        start += size - overlap
    return chunks

## 2. Build a simple metadata-aware index

Keep the document ID attached to every chunk so retrieved evidence can be cited.


In [ ]:
nodes = []
for document in documents:
    for index, chunk in enumerate(chunk_text(document["text"])):
        nodes.append({
            "node_id": f"{document['id']}#chunk-{index}",
            "document_id": document["id"],
            "text": chunk,
        })
print("indexed nodes:", nodes)

## 3. Retrieve nodes and cite sources

The actual LlamaIndex package can replace this simple overlap retriever; see `examples/integrations/llamaindex_example.py` for a credential-free framework demonstration.


In [ ]:
def query_index(question, top_k=2):
    terms = set(question.lower().split())
    ranked = sorted(
        ((len(terms & set(node["text"].lower().split())), node) for node in nodes),
        key=lambda item: -item[0],
    )
    return [node for score, node in ranked[:top_k] if score > 0]

hits = query_index("return products 30 days")
print([{"source": hit["node_id"], "text": hit["text"]} for hit in hits])

## 4. Optionally run the LlamaIndex framework

Install the `llamaindex` extra to execute the framework version with mock embeddings and an offline mock language model.


In [ ]:
RUN_LLAMAINDEX = False
if RUN_LLAMAINDEX:
    from examples.integrations.llamaindex_example import main as run_llamaindex

    run_llamaindex()
else:
    print("LlamaIndex framework call skipped; the local index above runs offline.")

## Wrap-up and next steps

- Identify what was learned, stored, retrieved, or generated in this implementation.
- Change an input and observe how the output changes.
- Test one failure case and decide when the system should abstain or ask for help.
- Replace toy data/components only after you can evaluate the change against a
  held-out or representative test set.

Continue with the [concept guide](../guides/concepts/057-llamaindex.md) and the
[learning roadmap](../guides/ROADMAP.md).
